# 03 — Fold-local, label-blind training

Each encoder starts from a recorded seed-matched initialization and is trained
only on its outer-training source videos. Sampling is source-uniform before a
sequence is selected, preventing long videos from dominating. Dataset folder
annotations do not enter the representation objective. Reflection augmentation
is a registered training variant, not a post-result repair.

Vanilla and reflection-augmented checkpoints use the same initialization,
source draws, target masks, geometric views, optimizer schedule, and update
count. Reflection has its own random stream, so consuming augmentation draws
cannot perturb sampling or masking. This makes their difference a paired
recipe ablation under the registered seeds—not a universal causal claim.

Checkpoint lineage binds protocol, cohort, split, allowed sources, fold, seed,
variant, and implementation. A mismatch fails instead of silently reusing a
stale model. `smoke` runs validate execution only; paper evidence requires the
locked paper profile, every registered fold and seed, and subsequent held-out
evaluation. Training cannot resolve the ethics or data-use gate.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display


def locate_suite_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for ancestor in (start, *start.parents):
        for candidate in (ancestor, ancestor / "neurips-laterality"):
            if (
                (candidate / "config" / "protocol.json").is_file()
                and (candidate / "laterality").is_dir()
            ):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate neurips-laterality from the current working directory."
    )


SUITE_ROOT = locate_suite_root()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))

from laterality.config import load_context

context = load_context(SUITE_ROOT / "config" / "protocol.json")
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")


def show_inline(figure):
    display(figure)
    plt.close(figure)


print(
    f"suite={SUITE_ROOT} profile={context.profile} "
    f"artifacts={context.artifact_root} protocol={context.protocol_digest[:12]}"
)

In [ ]:
import pandas as pd

from laterality.data import load_cohort
from laterality.splitting import load_splits
from laterality.training import train_selected
from laterality.visualization import training_figure

cohort = load_cohort(context)
splits = load_splits(context, cohort)
training_summaries = train_selected(context, cohort, splits)
show_inline(training_figure(context, training_summaries))
pd.DataFrame(training_summaries).drop(columns="history").sort_values(
    ["variant", "fold", "seed"]
).reset_index(drop=True)